In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os

from google.colab import files

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

print("Please upload your ZIP file.")

uploaded = files.upload()


# Get uploaded ZIP filename
zip_filename = list(uploaded.keys())[0]

print("\nUploaded file:")
print(zip_filename)


extract_path = "/content/dataset"

os.makedirs(
    extract_path,
    exist_ok=True
)

with zipfile.ZipFile(
    zip_filename,
    "r"
) as zip_ref:

    zip_ref.extractall(
        extract_path
    )


print("\nZIP file extracted successfully!")



train_file = None
test_file = None

for root, dirs, files_list in os.walk(extract_path):

    for file in files_list:

        if file.lower() == "train.csv":
            train_file = os.path.join(
                root,
                file
            )

        elif file.lower() == "test.csv":
            test_file = os.path.join(
                root,
                file
            )


# Check whether files exist
if train_file is None:

    raise FileNotFoundError(
        "train.csv was not found inside the ZIP file."
    )


if test_file is None:

    raise FileNotFoundError(
        "test.csv was not found inside the ZIP file."
    )


print("\nTraining file:")
print(train_file)

print("\nTesting file:")
print(test_file)



train_df = pd.read_csv(
    train_file
)

test_df = pd.read_csv(
    test_file
)


print("\nTraining data loaded!")
print("Training data shape:", train_df.shape)

print("\nTesting data loaded!")
print("Testing data shape:", test_df.shape)



print("\nFirst 5 rows of training data:")
print(train_df.head())


print("\nFirst 5 rows of testing data:")
print(test_df.head())


print("\nTraining columns:")
print(train_df.columns.tolist())


print("\nTesting columns:")
print(test_df.columns.tolist())


train_df.columns = train_df.columns.str.strip()

test_df.columns = test_df.columns.str.strip()


if "x" not in train_df.columns:

    raise ValueError(
        "Column 'x' is not present in train.csv"
    )


if "y" not in train_df.columns:

    raise ValueError(
        "Column 'y' is not present in train.csv"
    )


if "x" not in test_df.columns:

    raise ValueError(
        "Column 'x' is not present in test.csv"
    )


if "y" not in test_df.columns:

    raise ValueError(
        "Column 'y' is not present in test.csv"
    )

train_df = train_df.dropna(
    subset=["x", "y"]
)

test_df = test_df.dropna(
    subset=["x", "y"]
)


X_train = train_df[["x"]].values

y_train = train_df["y"].values.reshape(
    -1,
    1
)


X_test = test_df[["x"]].values

y_test = test_df["y"].values.reshape(
    -1,
    1
)


print("\nX_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_test = scaler.transform(
    X_test
)


X_train_b = np.c_[
    np.ones(
        (X_train.shape[0], 1)
    ),
    X_train
]


X_test_b = np.c_[
    np.ones(
        (X_test.shape[0], 1)
    ),
    X_test
]


print("\nX_train after adding bias:")
print(X_train_b[:5])


def gradient_descent(
    X,
    y,
    lr=0.01,
    n_iters=1000
):


    m = X.shape[0]

    # Number of features including bias
    n = X.shape[1]


    # Initialize weights
    theta = np.zeros(
        (n, 1)
    )


    # Store loss values
    losses = []


    for i in range(n_iters):


        y_pred = X @ theta


        error = y_pred - y

        loss = (
            1 / (2 * m)
        ) * np.sum(
            error ** 2
        )


        losses.append(
            loss
        )

        gradient = (
            1 / m
        ) * (
            X.T @ error
        )

        theta = (
            theta
            - lr * gradient
        )
        if i % 100 == 0:

            print(
                f"Iteration {i:4d} | "
                f"Loss: {loss:.6f}"
            )


    return theta, losses

print("\n")
print("=" * 60)
print("RUNNING GRADIENT DESCENT")
print("=" * 60)


theta, losses = gradient_descent(
    X_train_b,
    y_train,
    lr=0.01,
    n_iters=1000
)


y_pred = X_test_b @ theta


mse = mean_squared_error(
    y_test,
    y_pred
)


r2 = r2_score(
    y_test,
    y_pred
)


print("\n")
print("=" * 60)
print("GRADIENT DESCENT RESULTS")
print("=" * 60)


print("\nFinal weights (theta):")

print(
    theta.ravel()
)


print(
    f"\nTest MSE : {mse:.6f}"
)


print(
    f"Test R²  : {r2:.6f}"
)

print("\n")
print("=" * 60)
print("SCIKIT-LEARN LINEAR REGRESSION")
print("=" * 60)


sk_model = LinearRegression()


sk_model.fit(
    X_train,
    y_train.ravel()
)



sk_pred = sk_model.predict(
    X_test
)


sk_mse = mean_squared_error(
    y_test,
    sk_pred
)


sk_r2 = r2_score(
    y_test,
    sk_pred
)

print(
    f"\nSklearn MSE : {sk_mse:.6f}"
)


print(
    f"Sklearn R²  : {sk_r2:.6f}"
)

print("\n")
print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)


comparison = pd.DataFrame({

    "Model": [
        "Gradient Descent",
        "Scikit-learn Linear Regression"
    ],

    "MSE": [
        mse,
        sk_mse
    ],

    "R2 Score": [
        r2,
        sk_r2
    ]

})


print(
    comparison
)

prediction_df = pd.DataFrame({

    "Actual y": y_test.ravel(),

    "Gradient Descent Prediction":
        y_pred.ravel(),

    "Sklearn Prediction":
        sk_pred.ravel()

})


print("\n")
print("=" * 60)
print("SAMPLE PREDICTIONS")
print("=" * 60)


print(
    prediction_df.head(10)
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)


axes[0].plot(
    range(len(losses)),
    losses,
    color="steelblue"
)


axes[0].set_title(
    "Gradient Descent: Loss vs Iterations"
)


axes[0].set_xlabel(
    "Iteration"
)


axes[0].set_ylabel(
    "MSE Loss"
)


axes[0].grid(
    alpha=0.3
)


axes[1].scatter(
    y_test,
    y_pred,
    alpha=0.5,
    s=20,
    color="darkorange"
)

lims = [

    min(
        y_test.min(),
        y_pred.min()
    ),

    max(
        y_test.max(),
        y_pred.max()
    )

]


axes[1].plot(
    lims,
    lims,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Perfect prediction"
)


axes[1].set_title(
    "Actual vs Predicted Y"
)


axes[1].set_xlabel(
    "Actual Y"
)


axes[1].set_ylabel(
    "Predicted Y"
)


axes[1].legend()


axes[1].grid(
    alpha=0.3
)

plt.tight_layout()

plot_path = (
    "/content/gradient_descent_plots.png"
)


plt.savefig(
    plot_path,
    dpi=150
)


print("\n")
print(
    "Graph saved successfully:"
)


print(
    plot_path
)

plt.show()


print("\n")
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)


print(
    f"Training samples : {len(train_df)}"
)


print(
    f"Testing samples  : {len(test_df)}"
)


print(
    f"Gradient Descent MSE : {mse:.6f}"
)


print(
    f"Gradient Descent R²  : {r2:.6f}"
)


print(
    f"Sklearn MSE          : {sk_mse:.6f}"
)


print(
    f"Sklearn R²           : {sk_r2:.6f}"
)


print("\nProgram completed successfully!")

Please upload your ZIP file.
